# LC 994 — Rotting Oranges
**Day-67 | Theme: Multi-source BFS on Grids | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Seed BFS with <em>all</em> rotten oranges
at once. Every level of BFS represents one minute of simultaneous
rot spreading outward — this is classic multi-source BFS.
</div>

## Official Problem Statement

You are given an `m x n` grid where each cell can have one of three
values:
- `0` — empty cell
- `1` — fresh orange
- `2` — rotten orange

Every minute, any fresh orange that is **4-directionally adjacent**
to a rotten orange becomes rotten.

Return the **minimum number of minutes** that must elapse until no
cell has a fresh orange. If this is impossible, return `-1`.

**Constraints:**
- `m == grid.length`
- `n == grid[i].length`
- `1 <= m, n <= 10`
- `grid[i][j]` is `0`, `1`, or `2`

## What This Is Actually Asking

You have a grid of oranges — some fresh, some already rotten.
Rot spreads one step per minute to all 4 neighbors simultaneously.
You need to find how many minutes until all fresh oranges are rotten.
If any fresh orange is completely surrounded by empty cells and can
never be reached, return -1.
Think of it as a wildfire burning outward from multiple ignition
points at the same time.

## Walk Through an Example by Hand

```
Input grid:
  2 1 1
  1 1 0
  0 1 1
```

**Minute 0 (initial):** Rotten = {(0,0)}, Fresh count = 6

**Minute 1:** (0,0) spreads to (0,1) and (1,0)
```
  2 2 1
  2 1 0
  0 1 1
```
Fresh count = 4

**Minute 2:** (0,1) spreads to (0,2),(1,1); (1,0) already done
```
  2 2 2
  2 2 0
  0 1 1
```
Fresh count = 2

**Minute 3:** (1,1) spreads to (2,1)
```
  2 2 2
  2 2 0
  0 2 1
```
Fresh count = 1

**Minute 4:** (2,1) spreads to (2,2)
```
  2 2 2
  2 2 0
  0 2 2
```
Fresh count = 0  →  return **4**

## The Picture

Multi-source BFS waves spreading simultaneously from all rotten cells:

```
t=0          t=1          t=2          t=3          t=4
+-----+     +-----+     +-----+     +-----+     +-----+
|2 1 1|     |2 2 1|     |2 2 2|     |2 2 2|     |2 2 2|
|1 1 0|     |2 1 0|     |2 2 0|     |2 2 0|     |2 2 0|
|0 1 1|     |0 1 1|     |0 1 1|     |0 2 1|     |0 2 2|
+-----+     +-----+     +-----+     +-----+     +-----+
 seeds:R    wave 1       wave 2       wave 3       wave 4
 R=(0,0)   +(0,1)       +(0,2)       +(2,1)       +(2,2)
           +(1,0)       +(1,1)

BFS queue at t=0:  [(0,0)]
BFS queue at t=1:  [(0,1), (1,0)]          <- level boundary
BFS queue at t=2:  [(0,2), (1,1)]          <- level boundary
BFS queue at t=3:  [(2,1)]                 <- level boundary
BFS queue at t=4:  [(2,2)]  fresh=0  DONE

KEY: All rotten cells enqueued at t=0 — they fire in parallel!
     Each BFS level = 1 minute elapsed.
```

## When To Use This Pattern

- When **multiple sources** spread simultaneously, seed BFS with
  all of them at once — not one at a time.
- When you need **minimum time/steps** for something to propagate
  across a grid, think multi-source BFS.
- When the problem says "every cell does X to its neighbors at the
  same time", that is a BFS level = 1 unit of time signal.
- When you count fresh/unvisited items and decrement as BFS runs,
  leftover count > 0 means unreachable → return -1.
- When grid values encode state (0/1/2), BFS modifies state
  in-place to mark visited.

## The Approach

First, scan the grid once: add every rotten cell (value 2) to the
deque as initial seeds, and count all fresh cells (value 1).
Run BFS level by level — each level is one minute. For each rotten
cell dequeued, spread rot to 4-adjacent fresh cells, marking them
as rotten and decrementing the fresh count.
Track elapsed minutes by counting BFS levels (increment time after
processing each full level, only if the queue was non-empty).
After BFS ends, return `time` if `fresh == 0`, else `-1`.

In [ ]:
from typing import List
from collections import deque

In [ ]:
import copy


def test_harness(func):
    """
    Run test cases for LC 994 — Rotting Oranges.
    Uses copy.deepcopy so original grids are not mutated.
    """
    cases = [
        # (grid, expected, label)
        (
            [[2, 1, 1], [1, 1, 0], [0, 1, 1]],
            4,
            "basic spread",
        ),
        (
            [[2, 1, 1], [0, 1, 1], [1, 0, 1]],
            -1,
            "isolated fresh — impossible",
        ),
        (
            [[0, 2]],
            0,
            "no fresh oranges",
        ),
        (
            [[1]],
            -1,
            "single fresh, no rotten",
        ),
        (
            [[2]],
            0,
            "single rotten, no fresh",
        ),
        (
            [[2, 1]],
            1,
            "one rotten, one fresh neighbor",
        ),
        (
            [[0]],
            0,
            "all empty",
        ),
    ]

    passed = 0
    failed = 0
    for grid, expected, label in cases:
        result = func(copy.deepcopy(grid))
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            failed += 1
        print(
            f"{status} | {label:<35} "
            f"expected={expected} got={result}"
        )

    print(f"\nSummary: {passed} passed, {failed} failed "
          f"out of {passed + failed} tests")

In [ ]:
def oranges_rotting(grid: List[List[int]]) -> int:
    """
    Return the minimum minutes until all fresh oranges rot.

    Strategy: Multi-source BFS.
    - Seed the BFS queue with ALL rotten cells (value 2).
    - Count all fresh cells (value 1).
    - Process BFS level by level (each level = 1 minute).
    - Each rotten cell infects 4-adjacent fresh neighbors.
    - Return elapsed time if fresh count reaches 0, else -1.

    Args:
        grid: m x n grid with 0=empty, 1=fresh, 2=rotten.

    Returns:
        Minimum minutes to rot all oranges, or -1 if impossible.

    Examples:
        >>> oranges_rotting([[2,1,1],[1,1,0],[0,1,1]])
        4
        >>> oranges_rotting([[2,1,1],[0,1,1],[1,0,1]])
        -1
        >>> oranges_rotting([[0,2]])
        0
    """
    # Debug: print initial grid state
    print("[DEBUG] Initial grid:")
    for row in grid:
        print(" ", row)

    # TODO: Step 1 — scan grid, seed queue, count fresh
    # TODO: Step 2 — BFS level by level
    # TODO: Step 3 — return time or -1

    pass

In [ ]:
# Uncomment and run when solution is ready
# test_harness(oranges_rotting)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (simulate rounds) | O((m·n)²) | O(m·n) | Re-scan full grid each minute |
| Multi-source BFS (optimal) | O(m·n) | O(m·n) | Each cell visited at most once |

**m** = rows, **n** = cols.  
BFS queue holds at most all m·n cells — space is O(m·n).  
Each cell is enqueued and dequeued exactly once — time is O(m·n).

## Real World Connection

**Citi / Financial Systems:** Contagion modeling in risk — a single
defaulting counterparty can cascade to 4-degree neighbors in a
transaction graph; multi-source BFS computes the blast radius
and time-to-impact for stress tests.

**AWS / Cloud Infrastructure:** Propagating a config change or
security patch across a data-center network where nodes update
neighbors simultaneously; BFS levels map to deployment "waves."

**Data Engineering:** In a DAG of ETL jobs, if upstream failures
corrupt data, multi-source BFS from all failed nodes finds the
minimum number of pipeline stages before downstream tables are
impacted — critical for SLA breach analysis.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra